# Batfish Network Configuration Management Demo

**Batfish** is an open-source tool that builds a complete model of your network from device configuration files and lets you query it offline — no agents, no risk to production.

## What we cover in this notebook

1. **Setup** — Connect to Batfish, load a network snapshot
2. **Topology Exploration** — Discover devices, interfaces, OSPF/BGP sessions
3. **Reachability Analysis** — Verify which hosts can reach which services
4. **Routing Analysis** — Trace paths, detect black holes
5. **ACL Auditing** — Find shadowed rules, test specific flows
6. **Config Consistency** — Detect drift between devices
7. **Change Impact** — Quantify blast radius before deploying

## Network Topology

```
        Internet (203.0.113.0/30)
               |
         [firewall-01]
        /              \
[core-router-01]    [core-router-02]   <- iBGP AS65001 + OSPF Area 0
    /     \               /     \
[dist-sw-01]  --------  [dist-sw-02]
    |
[access-sw-01]
  /    \      \
Users  Users  Servers
```


## 1. Setup

In [ ]:
# Install pybatfish if not already installed
# !pip install pybatfish pandas tabulate

import os
import warnings
import pandas as pd
warnings.filterwarnings('ignore')

from pybatfish.client.session import Session
from pybatfish.datamodel import *
from pybatfish.datamodel.flow import HeaderConstraints, PathConstraints

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 200)

print('Imports OK')

In [ ]:
# Connect to the Batfish server
# If running via docker-compose, Batfish is at 'batfish' hostname
# If running Batfish separately, use 'localhost'
BATFISH_HOST = os.environ.get('BATFISH_HOST', 'batfish')

bf = Session(host=BATFISH_HOST)
bf.set_network('batfish-demo')
print(f'Connected to Batfish at {BATFISH_HOST}')

In [ ]:
# Load the 'good' network snapshot (correctly configured)
SNAPSHOTS_DIR = '../snapshots'

bf.init_snapshot(f'{SNAPSHOTS_DIR}/good-network', name='good-network', overwrite=True)
bf.set_snapshot('good-network')
print('Loaded: good-network snapshot')

## 2. Topology Exploration

Batfish automatically parses all configs and builds a network model. Let's explore it.

In [ ]:
# List all discovered devices
nodes = bf.q.nodeProperties(properties='Configuration_Format').answer().frame()
print(f'Devices discovered: {len(nodes)}')
nodes[['Node', 'Configuration_Format']]

In [ ]:
# List all active interfaces with IP addresses
ifaces = bf.q.interfaceProperties(
    properties='Interface,Primary_Address,Active,Description'
).answer().frame()

active = ifaces[ifaces['Active'] == True]
print(f'Active interfaces: {len(active)}')
active[['Interface', 'Primary_Address', 'Description']]

In [ ]:
# Layer-3 topology: which routers/switches are directly connected?
edges = bf.q.layer3Edges().answer().frame()
print(f'Layer-3 edges: {len(edges)}')
edges[['Interface', 'Remote_Interface']]

In [ ]:
# OSPF session compatibility
ospf = bf.q.ospfSessionCompatibility().answer().frame()
established = ospf[ospf['Session_Status'] == 'ESTABLISHED']
print(f'OSPF sessions: {len(established)} established, {len(ospf)-len(established)} broken')
ospf[['Interface', 'Remote_Interface', 'Session_Status']]

## 3. Reachability Analysis

Test whether specific traffic flows can traverse the network.
Batfish simulates every forwarding decision and reports ACCEPTED, DENIED, NULL_ROUTED, or NO_ROUTE.

In [ ]:
# Can users reach web servers via HTTP?
result = bf.q.reachability(
    headers=HeaderConstraints(
        srcIps='10.20.20.0/24',
        dstIps='10.10.10.0/24',
        ipProtocols=['TCP'],
        dstPorts=['80'],
    ),
    actions=['ACCEPTED', 'DENIED', 'NO_ROUTE', 'NULL_ROUTED'],
).answer().frame()

print('User -> Server HTTP reachability:')
print(result['Action'].value_counts())
result[['Flow', 'Action']].head(5)

In [ ]:
# Can the internet reach internal servers? (Should be DENIED)
internet_test = bf.q.reachability(
    headers=HeaderConstraints(
        srcIps='8.8.8.8/32',
        dstIps='10.10.10.50',
        ipProtocols=['TCP'],
        dstPorts=['80'],
    ),
    actions=['ACCEPTED', 'DENIED', 'NO_ROUTE'],
).answer().frame()

if len(internet_test) > 0:
    actions = internet_test['Action'].unique()
    if all(a in ['DENIED', 'NO_ROUTE'] for a in actions):
        print('PASS: Internet is correctly blocked from reaching servers')
    else:
        print('FAIL: Internet CAN reach servers! Security issue!')
else:
    print('No flows found')
internet_test[['Flow', 'Action']]

## 4. Routing Analysis

Inspect routing tables and trace forwarding paths hop-by-hop.

In [ ]:
# Routing table entries for the server subnet
routes = bf.q.routes(
    nodes='core-router-01|core-router-02',
    network='10.10.10.0/24'
).answer().frame()

print(f'Routes to 10.10.10.0/24 on core routers: {len(routes)}')
cols = [c for c in ['Node', 'Network', 'Protocol', 'Next_Hop_IP', 'Next_Hop_Interface', 'Admin_Distance'] if c in routes.columns]
routes[cols]

In [ ]:
# Traceroute from user to server
trace = bf.q.traceroute(
    startLocation='@enter(dist-switch-01[Vlan20])',
    headers=HeaderConstraints(
        srcIps='10.20.20.100',
        dstIps='10.10.10.50',
        ipProtocols=['TCP'],
        dstPorts=['443'],
    ),
).answer().frame()

for _, row in trace.iterrows():
    print(f"Flow: {row.get('Flow', '')}")
    for t in list(row.get('Traces', []))[:1]:
        print(f'  Disposition: {t.disposition}')
        for i, hop in enumerate(t.hops):
            print(f'  Hop {i+1}: {hop.node}')

In [ ]:
# Check for routing loops
loops = bf.q.detectLoops().answer().frame()
if len(loops) == 0:
    print('No routing loops detected.')
else:
    print(f'WARNING: {len(loops)} routing loops found!')
    print(loops)

## 5. ACL Analysis

Batfish can find shadowed ACL rules (rules that can never match because an earlier rule catches all relevant packets). This is very hard to find manually in complex ACLs.

In [ ]:
# Check for shadowed rules in good network
bf.set_snapshot('good-network')

acl_reach = bf.q.filterLineReachability(
    filters='SERVER_ACL',
    nodes='dist-switch-01',
).answer().frame()

print('Good network - dist-switch-01 SERVER_ACL line reachability:')
print(acl_reach.to_string(index=False))

In [ ]:
# Load the bad network snapshot (with 4 intentional bugs)
bf.init_snapshot(f'{SNAPSHOTS_DIR}/bad-network', name='bad-network', overwrite=True)
bf.set_snapshot('bad-network')
print('Loaded: bad-network snapshot')
print()
print('This snapshot has 4 intentional bugs:')
print('  Bug 1: Null0 black-hole route on core-router-01')
print('  Bug 2: Shadowed ACL rules on dist-switch-01')
print('  Bug 3: Inconsistent ACL on dist-switch-02 (missing SSH rule)')
print('  Bug 4: Firewall allows all internet traffic (permit any any first)')

In [ ]:
# Find shadowed rules in bad network
acl_bad = bf.q.filterLineReachability(
    filters='SERVER_ACL',
    nodes='dist-switch-01',
).answer().frame()

print('Bad network - dist-switch-01 SERVER_ACL:')
unreachable_col = next((c for c in acl_bad.columns if 'Unreachable' in c), None)
if unreachable_col:
    shadowed = acl_bad[acl_bad[unreachable_col].notna()]
    print(f'Shadowed lines found: {len(shadowed)}')
print(acl_bad.to_string(index=False))

## 6. Configuration Consistency

In [ ]:
# Check for undefined references (e.g., ACL applied to interface but not defined)
undef = bf.q.undefinedReferences().answer().frame()
print(f'Undefined references: {len(undef)}')
if len(undef) > 0:
    print(undef.to_string(index=False))

# Check for unused structures
unused = bf.q.unusedStructures().answer().frame()
print(f'Unused structures: {len(unused)}')
if len(unused) > 0:
    print(unused.to_string(index=False))

## 7. Pre-Change Impact Analysis

Before deploying a config change, Batfish can tell you exactly which traffic flows will be added or removed.
This is the "blast radius" calculation that makes Batfish invaluable in CI/CD pipelines.

In [ ]:
import shutil, os

CANDIDATE_PATH = f'{SNAPSHOTS_DIR}/candidate-change'

# Create candidate snapshot: copy good-network and add a new deny-ICMP rule
if os.path.exists(CANDIDATE_PATH):
    shutil.rmtree(CANDIDATE_PATH)
shutil.copytree(f'{SNAPSHOTS_DIR}/good-network', CANDIDATE_PATH)

cfg = f'{CANDIDATE_PATH}/configs/dist-switch-01.cfg'
with open(cfg) as f: content = f.read()

# Insert deny ICMP rule at the top of SERVER_ACL
content = content.replace(
    'ip access-list extended SERVER_ACL',
    'ip access-list extended SERVER_ACL\n deny   icmp 10.20.20.0 0.0.0.255 10.10.10.0 0.0.0.255 log',
    1
)
with open(cfg, 'w') as f: f.write(content)

# Load candidate into Batfish
bf.init_snapshot(CANDIDATE_PATH, name='candidate-change', overwrite=True)
print('Candidate snapshot loaded.')
print('Proposed change: add deny-ICMP rule to dist-switch-01 SERVER_ACL')

In [ ]:
# What traffic flows will be LOST by this change?
bf.set_snapshot('good-network')

lost = bf.q.differentialReachability(
    headers=HeaderConstraints(
        srcIps='10.20.20.0/24',
        dstIps='10.10.10.0/24',
    ),
).answer(
    snapshot='good-network',
    reference_snapshot='candidate-change',
).frame()

print(f'Traffic flows that will be BLOCKED after the change: {len(lost)}')
if len(lost) > 0:
    print('These flows currently work but will break:')
    flow_col = 'Flow' if 'Flow' in lost.columns else lost.columns[0]
    for _, row in lost.head(5).iterrows():
        print(f'  {row.get(flow_col, row)}')
else:
    print('No currently-working flows will be broken.')

In [ ]:
# Cleanup
if os.path.exists(CANDIDATE_PATH):
    shutil.rmtree(CANDIDATE_PATH)
    print('Cleaned up candidate snapshot directory.')

## Summary

Batfish found all 4 intentional bugs in the `bad-network` snapshot:

| Bug | Location | Detection Method |
|---|---|---|
| Null0 black-hole route | `core-router-01` | `routes()` + `reachability()` |
| Shadowed ACL rules | `dist-switch-01` | `filterLineReachability()` |
| Inconsistent SSH ACL | `dist-switch-02` | `searchFilters()` + `reachability()` |
| Firewall allows all traffic | `firewall-01` | `filterLineReachability()` |

### Key Takeaways

- Batfish analysis is **100% offline** — no need to touch live devices
- All major vendors supported: Cisco IOS/NX-OS, Juniper, Arista, Palo Alto, F5, and more
- Integrates into **CI/CD pipelines** to catch bugs before deployment
- `differentialReachability()` quantifies the exact blast radius of any change
- Python API returns **pandas DataFrames** — easy to integrate with reporting tools
